# Qwen2-Audio baseline and int8 quantization
Starter notebook.


In [1]:
# ============================================================
# CELL 1: Imports and global config
# ============================================================

from pathlib import Path
import shutil
import csv
import time

import pandas as pd
import soundfile as sf
import torch
from tqdm.auto import tqdm

from transformers import (
    AutoProcessor,
    Qwen2AudioForConditionalGeneration,
    BitsAndBytesConfig,
)

from comet import download_model, load_from_checkpoint


PROJECT_ROOT = Path("/home/paperspace/projects/iwslt2026-compression")
RAW_ROOT = PROJECT_ROOT / "data" / "raw" / "acl60_60" / "2" / "acl_6060"
FILTERED_ROOT = PROJECT_ROOT / "data" / "raw" / "acl6060_en_zh"
MANIFEST_DIR = PROJECT_ROOT / "data" / "manifests"
OUT_DIR = PROJECT_ROOT / "outputs" / "qwen2audio_full_en_zh"

DEV_MANIFEST = MANIFEST_DIR / "acl6060_dev_en_zh.csv"
EVAL_MANIFEST = MANIFEST_DIR / "acl6060_eval_en_zh.csv"

MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
PROMPT_TEXT = "Translate this English speech into Chinese."

DEVICE = "cuda:0"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("project root exists:", PROJECT_ROOT.exists())
print("raw root exists:", RAW_ROOT.exists())
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


# ============================================================
# CELL 2: Build manifests if missing, then load full dataset
# ============================================================

def rebuild_en_zh_subset_and_manifests():
    print("Rebuilding en->zh subset and manifests...")

    if FILTERED_ROOT.exists():
        shutil.rmtree(FILTERED_ROOT)

    for split in ["dev", "eval"]:
        (FILTERED_ROOT / split / "text" / "txt").mkdir(parents=True, exist_ok=True)
        (FILTERED_ROOT / split / "segmented_wavs").mkdir(parents=True, exist_ok=True)

        shutil.copy2(
            RAW_ROOT / split / "text" / "txt" / f"ACL.6060.{split}.en-xx.en.txt",
            FILTERED_ROOT / split / "text" / "txt" / f"ACL.6060.{split}.en-xx.en.txt",
        )
        shutil.copy2(
            RAW_ROOT / split / "text" / "txt" / f"ACL.6060.{split}.en-xx.zh.txt",
            FILTERED_ROOT / split / "text" / "txt" / f"ACL.6060.{split}.en-xx.zh.txt",
        )
        shutil.copytree(
            RAW_ROOT / split / "segmented_wavs" / "gold",
            FILTERED_ROOT / split / "segmented_wavs" / "gold",
        )

    MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

    for split in ["dev", "eval"]:
        en_file = FILTERED_ROOT / split / "text" / "txt" / f"ACL.6060.{split}.en-xx.en.txt"
        zh_file = FILTERED_ROOT / split / "text" / "txt" / f"ACL.6060.{split}.en-xx.zh.txt"
        wav_dir = FILTERED_ROOT / split / "segmented_wavs" / "gold"

        en_lines = en_file.read_text(encoding="utf-8").splitlines()
        zh_lines = zh_file.read_text(encoding="utf-8").splitlines()
        wavs = sorted(wav_dir.glob("sent_*.wav"), key=lambda p: int(p.stem.split("_")[1]))

        assert len(en_lines) == len(zh_lines) == len(wavs), (
            split, len(en_lines), len(zh_lines), len(wavs)
        )

        out_path = MANIFEST_DIR / f"acl6060_{split}_en_zh.csv"
        with out_path.open("w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["id", "split", "audio_path", "source_en", "target_zh"])
            for i, (wav, en, zh) in enumerate(zip(wavs, en_lines, zh_lines), start=1):
                writer.writerow([f"{split}_{i}", split, str(wav.relative_to(PROJECT_ROOT)), en, zh])

        print("wrote", out_path, "rows:", len(wavs))


if not DEV_MANIFEST.exists() or not EVAL_MANIFEST.exists():
    rebuild_en_zh_subset_and_manifests()

df = pd.concat(
    [
        pd.read_csv(DEV_MANIFEST),
        pd.read_csv(EVAL_MANIFEST),
    ],
    ignore_index=True,
)

df["audio_path"] = df["audio_path"].apply(
    lambda p: str((PROJECT_ROOT / p).resolve()) if not Path(p).is_absolute() else str(Path(p).resolve())
)

print("total rows:", len(df))
print("first audio path:", df["audio_path"].iloc[0])
print("first audio exists:", Path(df["audio_path"].iloc[0]).exists())

df.head()


# ============================================================
# CELL 3: Tiny smoke subset
# Change USE_FULL_DATASET to True when ready
# ============================================================

USE_FULL_DATASET = True
SMOKE_N = 8

run_df = df.copy() if USE_FULL_DATASET else df.head(SMOKE_N).copy()
print("rows selected for run:", len(run_df))


# ============================================================
# CELL 4: Load processor, baseline model, and int8 model
# ============================================================

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID)

print("Loading baseline fp16 model...")
baseline_model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
baseline_model.eval()

print("Loading int8 model...")
bnb_config = BitsAndBytesConfig(load_in_8bit=True)
quant_model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
quant_model.eval()

print("Models loaded.")


# ============================================================
# CELL 5: Inference helper with progress bar
# ============================================================

def generate_translation(model, processor, audio_path, prompt_text=PROMPT_TEXT, max_new_tokens=128):
    audio, sr = sf.read(str(audio_path))

    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "audio", "audio_url": str(audio_path)},
                {"type": "text", "text": prompt_text},
            ],
        }
    ]

    text = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=False,
    )

    inputs = processor(
        text=text,
        audios=[audio],
        sampling_rate=sr,
        return_tensors="pt",
    )

    inputs = {
        k: v.to(DEVICE) if hasattr(v, "to") else v
        for k, v in inputs.items()
    }

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)

    generated_ids = generated_ids[:, inputs["input_ids"].size(1):]

    pred = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    return pred


def run_inference(model, processor, df_input, run_name):
    preds = []
    start = time.time()

    for _, row in tqdm(df_input.iterrows(), total=len(df_input), desc=run_name):
        audio_path = Path(row["audio_path"])
        pred = generate_translation(model, processor, audio_path)
        preds.append(pred)

    elapsed = time.time() - start

    out_df = df_input.copy()
    out_df["prediction"] = preds
    return out_df, elapsed


# ============================================================
# CELL 6: Run baseline and int8, save predictions
# ============================================================

print("Running baseline fp16...")
baseline_df, baseline_time = run_inference(
    baseline_model,
    processor,
    run_df,
    run_name="baseline_fp16",
)
baseline_out = OUT_DIR / "baseline_full_preds.csv"
baseline_df.to_csv(baseline_out, index=False, encoding="utf-8")
print("baseline saved to:", baseline_out)
print("baseline seconds:", round(baseline_time, 2))

print("Running int8...")
quant_df, quant_time = run_inference(
    quant_model,
    processor,
    run_df,
    run_name="quantized_int8",
)
quant_out = OUT_DIR / "quant8_full_preds.csv"
quant_df.to_csv(quant_out, index=False, encoding="utf-8")
print("int8 saved to:", quant_out)
print("int8 seconds:", round(quant_time, 2))

baseline_df[["id", "target_zh", "prediction"]].head(3)


# ============================================================
# CELL 7: Load COMET and score both runs
# ============================================================

print("Downloading/loading COMET...")
comet_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(comet_path)

print("Preparing COMET data for baseline...")
baseline_comet_data = [
    {"src": r["source_en"], "mt": r["prediction"], "ref": r["target_zh"]}
    for _, r in tqdm(baseline_df.iterrows(), total=len(baseline_df), desc="comet_prep_baseline")
]

print("Preparing COMET data for int8...")
quant_comet_data = [
    {"src": r["source_en"], "mt": r["prediction"], "ref": r["target_zh"]}
    for _, r in tqdm(quant_df.iterrows(), total=len(quant_df), desc="comet_prep_int8")
]

print("Scoring baseline with COMET...")
baseline_comet_out = comet_model.predict(baseline_comet_data, batch_size=4, gpus=1)
baseline_comet = baseline_comet_out.system_score if hasattr(baseline_comet_out, "system_score") else baseline_comet_out[1]

print("Scoring int8 with COMET...")
quant_comet_out = comet_model.predict(quant_comet_data, batch_size=4, gpus=1)
quant_comet = quant_comet_out.system_score if hasattr(quant_comet_out, "system_score") else quant_comet_out[1]

print("baseline COMET:", baseline_comet)
print("int8 COMET:", quant_comet)


# ============================================================
# CELL 8: Summary and save
# ============================================================

summary = pd.DataFrame(
    [
        {
            "run": "baseline_fp16",
            "rows": len(baseline_df),
            "seconds": baseline_time,
            "comet": baseline_comet,
        },
        {
            "run": "quantized_int8",
            "rows": len(quant_df),
            "seconds": quant_time,
            "comet": quant_comet,
        },
    ]
)

baseline_score = summary.loc[summary["run"] == "baseline_fp16", "comet"].iloc[0]
summary["comet_diff_vs_baseline"] = summary["comet"] - baseline_score
summary["speedup_vs_baseline"] = baseline_time / summary["seconds"]

summary_out = OUT_DIR / "summary_comet_only.csv"
summary.to_csv(summary_out, index=False)

print("summary saved to:", summary_out)
summary

project root exists: True
raw root exists: True
cuda available: True
gpu: NVIDIA RTX A6000
total rows: 884
first audio path: /home/paperspace/projects/iwslt2026-compression/data/raw/acl6060_en_zh/dev/segmented_wavs/gold/sent_1.wav
first audio exists: True
rows selected for run: 884
Loading processor...
Loading baseline fp16 model...


We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Loading int8 model...


We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


...

baseline_fp16:   0%|          | 0/884 [00:00<?, ?it/s]

We detected that you are passing `past_key_values` as a tuple of tuples. This is deprecated and will be removed in v4.47. Please convert your cache or use an appropriate `Cache` class (https://huggingface.co/docs/transformers/kv_cache#legacy-cache-format)


baseline saved to: /home/paperspace/projects/iwslt2026-compression/outputs/qwen2audio_full_en_zh/baseline_full_preds.csv
baseline seconds: 577.26
Running int8...


quantized_int8:   0%|          | 0/884 [00:00<?, ?it/s]

KeyboardInterrupt: 